# 🖥️ Delft3D tutorial

This tutorial shows how to load in [Delft3D](https://www.deltares.nl/en/software/delft3d-4-suite/) model output files (in NetCDF format) into Parcels.

## Structured Grids
Special about Delft3D is that its structured grid contains NaN values for points that don't exist in the model domain. The hashtable approach in Parcels v4 supports these NaN grid cells out of the box, so that you can use the model output directly without any preprocessing.

In [ ]:
import cmocean as cmo
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import parcels
import parcels.tutorial

The example below is of a small domain in the Port of Rotterdam. We first pick the coorindates and the velocity fields from the dataset, and then convert them to the sgrid conventions. Finally, we create a FieldSet from the converted dataset.

```{note}
While Delft3D provides velocities on a CGrid, we currently can't use the {py:obj}`parcels.interpolators.CGrid_Velocity` interpolator on the velocity fields. This is because the velocities have been rotated to east and north in the output - but the Parcels interpolator expects them in the `M` and `N` (along-grid) directions. Therefore, we use the {py:obj}`parcels.interpolators.XFreeslip` interpolator instead.
```

In [ ]:
ds = parcels.tutorial.open_dataset("Delft3D_data/Rotterdam_tiny")
coords = ds[["XZETA", "YZETA", "SIGMA_C"]]
ds_fset = parcels.convert.delft3d_to_sgrid(
    fields={"U": ds["VELU"], "V": ds["VELV"]}, coords=coords
)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)

# Set the interpolation method for the UV field to XFreeslip (see note above)
fieldset.UV.interp_method = parcels.interpolators.XFreeslip()

fieldset = fieldset.to_windowed_arrays()
fieldset.describe()

Now we define a grid of a few particles to release in the domain, and run a simple advection simulation. The particles are advected by the Delft3D velocity fields, and we can visualize their trajectories.

In [ ]:
# Define a set of starting points
points_x, points_y = np.meshgrid(
    np.linspace(93000, 93300, 5), np.linspace(436300, 436600, 5)
)
z = np.full(points_x.shape, 0.06)  # Depth of 0.06 m
pset = parcels.ParticleSet(fieldset, x=points_x, y=points_y, z=z)

# St up an output file to save particle trajectories
outputfile = parcels.ParticleFile(
    "Delft3D_structured.parquet",
    outputdt=np.timedelta64(60, "s"),
    mode="w",
)


# Define a custom error handling kernel to delete particles on any error
def DeleteOnAnyError(particles, fieldset):
    any_error = particles.state >= 50  # This captures all Errors
    particles[any_error].state = parcels.StatusCode.Delete


# Run the particle set with the advection kernel and the custom error handling kernel
pset.execute(
    [parcels.kernels.AdvectionRK2, DeleteOnAnyError],
    endtime=fieldset.time_interval.right,
    dt=np.timedelta64(60, "s"),
    output_file=outputfile,
    verbose_progress=False,
)

In [ ]:
df = parcels.read_particlefile("Delft3D_structured.parquet")

fig, ax = plt.subplots(figsize=(8, 6))

# Plot background velocity field
time_idx = 0
layer_idx = 0
speed = (
    np.sqrt(ds["VELV"] ** 2 + ds["VELU"] ** 2)
    .isel(TIME=time_idx, LAYER=layer_idx)
    .compute()
)
tpc = plt.pcolor(
    speed.XZETA.values,
    speed.YZETA.values,
    speed,
    edgecolors="black",
    cmap="cmo.speed",
    vmin=0,
    vmax=np.max(np.abs(speed)),
)
time_str = pd.to_datetime(speed.TIME.values).strftime("%Y-%m-%d %H:%M:%S")
fig.colorbar(
    tpc,
    ax=ax,
    label=f"Flow speed on {time_str} and {speed.SIGMA_C.values:.2f}m depth [m/s]",
)

for traj in df.partition_by("particle_id"):
    ax.plot(traj["x"], traj["y"], "b", alpha=0.8)

ax.plot(points_x, points_y, "m.", label="initial particle positions")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel(ds.XZETA.long_name)
ax.set_ylabel(ds.YZETA.long_name)
plt.show()